# Bab 8 — Persiapan dan distribusi data RT-IoT2022

Jalankan sel berurutan; letakkan RT_IOT2022.csv sesuai Kode 8.1. Keluaran masuk folder output/.

## 📸 SCREENSHOT — Gambar 8.2 (satu-satunya gambar yang harus Anda ambil sendiri)

Setelah semua sel di bawah selesai dijalankan di Google Colab, ambil SATU tangkapan layar yang memperlihatkan: (1) sebuah sel kode beserta keluarannya, dan (2) panel Berkas di kiri dengan folder `data/` dan `output/` terbuka. Simpan sebagai PNG (lebar sekitar 15 cm, 300 dpi), lalu sisipkan di Word pada kotak abu-abu "Gambar 8.2" (Bab 8). Gambar lain sudah dibuat dari data nyata.

## Kode 8.1

In [ ]:
!pip install -q imbalanced-learn xgboost

import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, imblearn, xgboost

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Struktur folder kerja
for folder in ("data", "output"):
    os.makedirs(folder, exist_ok=True)

# Catat versi (untuk bagian reprodusibilitas)
print("Python", sys.version.split()[0])
for lib in (np, pd, sklearn, imblearn, xgboost):
    print(lib.__name__, lib.__version__)

## Kode 8.2

In [ ]:
NAMA_FILE = "data/RT_IOT2022.csv"   # CEK: sesuaikan dengan nama berkas hasil unduhan Anda

df = pd.read_csv(NAMA_FILE)
print("Bentuk data (baris, kolom):", df.shape)
print(df.head().T)        # .T agar puluhan kolom terbaca ke bawah
df.info()
print(df.describe().T)

## Kode 8.3

In [ ]:
LABEL = "Attack_type"     # CEK: nama kolom label menurut halaman dataset
assert LABEL in df.columns, "Kolom label tidak ditemukan; periksa df.columns"

print("Kolom label  :", LABEL)
print("Jumlah kelas :", df[LABEL].nunique())
print("Daftar label  :", sorted(df[LABEL].unique()))

# Kandidat kolom yang bukan fitur:
#   - nilainya unik di setiap baris (indeks atau identitas)
#   - nilainya konstan (tidak membedakan apa pun)
unik_semua = [c for c in df.columns if c != LABEL and df[c].nunique() == len(df)]
konstan = [c for c in df.columns if c != LABEL and df[c].nunique() <= 1]
print("Unik di setiap baris:", unik_semua)
print("Konstan              :", konstan)

# Kolom "Unnamed: n" muncul bila CSV disimpan bersama indeks pandas. Ia belum tentu unik di
# setiap baris (mis. indeks yang mulai dari 0 lagi di tiap kelas), jadi periksa per kelas.
kolom_indeks = [c for c in df.columns if str(c).startswith("Unnamed")]
for c in kolom_indeks:
    print(f"Kolom indeks '{c}': nilai per kelas")
    print(df.groupby(LABEL)[c].agg(["min", "max", "count", "nunique"]))

## Kode 8.4

In [ ]:
# a) Buang kolom bukan-fitur lebih dulu. Kolom indeks membuat setiap baris
#    tampak unik, sehingga duplikat tidak akan terdeteksi bila kolom itu dibiarkan.
KOLOM_DIBUANG = list(dict.fromkeys(unik_semua + kolom_indeks + konstan))   # CEK sebelum dijalankan
df = df.drop(columns=KOLOM_DIBUANG)
print("Kolom dibuang:", KOLOM_DIBUANG, "| bentuk sekarang:", df.shape)

# b) Nilai kosong
kosong = df.isna().sum()
print("Kolom dengan nilai kosong:")
print(kosong[kosong > 0])
print("Total nilai kosong:", int(kosong.sum()))

# c) Duplikasi
n_dup = int(df.duplicated().sum())
print(f"Baris duplikat: {n_dup} ({n_dup / len(df):.2%})")

# d) Duplikat yang berlabel berbeda (pola fitur sama, label berlainan)
fitur = [c for c in df.columns if c != LABEL]
label_per_pola = df.groupby(fitur, dropna=False)[LABEL].nunique()
print("Pola fitur dengan lebih dari satu label:", int((label_per_pola > 1).sum()))

# e) Tipe data
print(df.dtypes.value_counts())
kolom_kategorikal = [c for c in df.select_dtypes(include=["object", "category"]).columns
                     if c != LABEL]
print("Kolom kategorikal (selain label):", kolom_kategorikal)

# f) Nilai pengganti yang bukan NaN (mis. "-") tidak terdeteksi isna(); lihat nilai teratas
for c in kolom_kategorikal:
    print(c, df[c].value_counts().head(5).to_dict())

## Kode 8.5

In [ ]:
# Simpan distribusi sebelum duplikat dibuang, lalu buang duplikat SEBELUM pembagian data
dist_sebelum = df[LABEL].value_counts()
df = df.drop_duplicates().reset_index(drop=True)
print("Bentuk setelah pembersihan:", df.shape)

# Distribusi kelas, persentase, dan imbalance ratio (IR) per kelas
dist = df[LABEL].value_counts()
tabel = pd.DataFrame({
    "jumlah": dist,
    "persen": (dist / len(df) * 100).round(3),
    "IR": (dist.max() / dist).round(2),
    "dibuang_duplikat": (dist_sebelum - dist).reindex(dist.index),
})
print(tabel)
print("IR keseluruhan (terbesar / terkecil):", round(dist.max() / dist.min(), 2))
tabel.to_csv("output/dist_kelas.csv")

# Grafik: skala linear dan logaritmik berdampingan
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
dist.plot.barh(ax=ax[0], color="#2E9C93");  ax[0].set_title("Skala linear")
dist.plot.barh(ax=ax[1], color="#E07B39");  ax[1].set_title("Skala logaritmik")
ax[1].set_xscale("log")
for a in ax:
    a.invert_yaxis()
    a.set_xlabel("Jumlah sampel")
plt.tight_layout()
plt.savefig("output/g_distribusi.png", dpi=200)
plt.show()

## Kode 8.6

In [ ]:
from sklearn.preprocessing import LabelEncoder

X = df.drop(columns=[LABEL])
y_teks = df[LABEL]

# Label multikelas: teks -> angka 0..K-1 (diperlukan XGBoost). Simpan pemetaannya.
le = LabelEncoder()
y = le.fit_transform(y_teks)
pemetaan = pd.DataFrame({"kode": range(len(le.classes_)), "label": le.classes_})
pemetaan.to_csv("output/pemetaan_label.csv", index=False)
print(pemetaan)

# Label biner: normal (0) vs serangan (1), dibentuk dari label multikelas.
# Nama kelas normal belum pasti, jadi kelas serangan dikenali dari kata kunci.
KATA_KUNCI_SERANGAN = ("DOS", "DDOS", "ARP", "NMAP", "Metasploit", "Brute")   # CEK
serangan = [k for k in le.classes_ if any(s.lower() in k.lower() for s in KATA_KUNCI_SERANGAN)]
normal = [k for k in le.classes_ if k not in serangan]
print("Dianggap serangan:", serangan)     # PERIKSA: harus sesuai daftar serangan
print("Dianggap normal  :", normal)       # PERIKSA: harus lalu lintas normal
y_biner = y_teks.isin(serangan).astype(int).to_numpy()
print("Biner (0 = normal, 1 = serangan):", {int(k): int(v) for k, v in zip(*np.unique(y_biner, return_counts=True))})

## Kode 8.7

In [ ]:
from sklearn.model_selection import train_test_split

# Periksa dulu: stratifikasi memerlukan minimal 2 sampel per kelas
jumlah_min = pd.Series(y).value_counts().min()
assert jumlah_min >= 2, "Ada kelas dengan kurang dari 2 sampel; lihat pembahasan kelas langka"

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
# Label biner dibagi dengan indeks yang sama agar konsisten dengan label multikelas
y_train_biner = y_biner[X_train.index]
y_test_biner = y_biner[X_test.index]

# Verifikasi proporsi tiap kelas pada data latih dan data uji
cek = pd.DataFrame({
    "latih": pd.Series(y_train).value_counts().sort_index(),
    "uji": pd.Series(y_test).value_counts().sort_index(),
})
cek["persen_latih"] = (cek["latih"] / cek["latih"].sum() * 100).round(3)
cek["persen_uji"] = (cek["uji"] / cek["uji"].sum() * 100).round(3)
cek.index = le.classes_[cek.index]
print(cek)
cek.to_csv("output/proporsi_latih_uji.csv")
print("Sampel kelas terkecil pada data latih:", int(cek["latih"].min()))

## Kode 8.7b

In [ ]:
# Ilustrasi: kelas dengan 1 sampel tidak dapat distratifikasi (DATA SINTETIS)
import numpy as np
from sklearn.model_selection import train_test_split

y_kecil = np.array([0] * 98 + [1] * 1 + [2] * 1)
X_kecil = np.arange(100).reshape(-1, 1)
try:
    train_test_split(X_kecil, y_kecil, test_size=0.3, stratify=y_kecil, random_state=42)
except ValueError as e:
    print("ValueError:", str(e)[:140])

## Kode 8.8

In [ ]:
# Ilustrasi kebocoran data pada DATA SINTETIS (bukan RT-IoT2022)
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
X, y = make_classification(n_samples=4000, n_features=10, n_informative=4,
    n_redundant=2, weights=[0.95, 0.05], class_sep=0.5, flip_y=0.02,
    random_state=RANDOM_STATE)

def skor(model, X_uji, y_uji):
    p = model.predict(X_uji)
    return recall_score(y_uji, p), f1_score(y_uji, p)

# SALAH: SMOTE pada seluruh data, baru dibagi
X_s, y_s = SMOTE(random_state=RANDOM_STATE).fit_resample(X, y)
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
    X_s, y_s, test_size=0.30, stratify=y_s, random_state=RANDOM_STATE)
m_salah = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
m_salah.fit(Xa_tr, ya_tr)
r1, f1 = skor(m_salah, Xa_te, ya_te)

# BENAR: dibagi dulu, SMOTE hanya pada data latih, uji tetap asli
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
Xb_bal, yb_bal = SMOTE(random_state=RANDOM_STATE).fit_resample(Xb_tr, yb_tr)
m_benar = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
m_benar.fit(Xb_bal, yb_bal)
r2, f2 = skor(m_benar, Xb_te, yb_te)

print(f"Data uji SALAH : n={len(ya_te)}, minoritas={int(ya_te.sum())} ({ya_te.mean():.0%})")
print(f"Data uji BENAR : n={len(yb_te)}, minoritas={int(yb_te.sum())} ({yb_te.mean():.1%})")
print(f"SALAH (SMOTE sebelum split): recall minoritas={r1:.3f}  F1 minoritas={f1:.3f}")
print(f"BENAR (split dulu)          : recall minoritas={r2:.3f}  F1 minoritas={f2:.3f}")

## Kode 8.9

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

kolom_numerik = [c for c in X_train.columns if c not in kolom_kategorikal]

# Semua langkah yang "belajar dari data" berada di dalam satu pipeline
praproses = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), kolom_numerik),
    ("kat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
     kolom_kategorikal),
])

pipa = Pipeline([
    ("praproses", praproses),
    ("sampler", RandomUnderSampler(random_state=RANDOM_STATE)),   # contoh; diganti di Bab 9
    ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
])

# Validasi silang pada DATA LATIH: sampler hanya aktif pada lipatan latih
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
skor_cv = cross_val_score(pipa, X_train, y_train, cv=cv, scoring="f1_macro")
print("F1-macro per lipatan:", skor_cv.round(3), "| rata-rata:", skor_cv.mean().round(3))

## Kode 8.11

In [ ]:
# Simpan data hasil pembagian agar dapat dipakai ulang di Bab 9 sampai Bab 12
X_train.to_csv("output/X_train.csv", index=False)
X_test.to_csv("output/X_test.csv", index=False)
pd.Series(y_train, name="y").to_csv("output/y_train.csv", index=False)
pd.Series(y_test, name="y").to_csv("output/y_test.csv", index=False)
pd.Series(y_train_biner, name="y_biner").to_csv("output/y_train_biner.csv", index=False)
pd.Series(y_test_biner, name="y_biner").to_csv("output/y_test_biner.csv", index=False)

# Catat lingkungan dan keputusan agar eksperimen dapat diulang
with open("output/lingkungan.txt", "w") as f:
    f.write(f"Python {sys.version.split()[0]}\n")
    for lib in (np, pd, sklearn, imblearn, xgboost):
        f.write(f"{lib.__name__} {lib.__version__}\n")
    f.write(f"RANDOM_STATE = {RANDOM_STATE}\n")
    f.write(f"Kolom dibuang: {KOLOM_DIBUANG}\n")
    f.write(f"Kolom kategorikal: {kolom_kategorikal}\n")
    f.write(f"Baris setelah pembersihan: {len(df)}\n")

print("Berkas di folder output:", sorted(os.listdir("output")))

## 📸 SCREENSHOT SEKARANG — Gambar 8.2

Buka panel Berkas (ikon folder, kiri), perluas `data/` dan `output/`, lalu ambil tangkapan layar bersama sel Kode 8.1 atau 8.3 beserta keluarannya.